# «Середины квадратов»

# Генерация последовательности

In [1]:
print("Ведите числа:")
print("1) k для X0 = 2 + 4 * k")
print("2) n для кол-ва чисел")
print("3) e для Xn+1 = Xn * (Xn + 1) mod 2^e")

A = [int(i) for i in input().split()]  # k N e

print(" >>>>>----------------------------------------------------->")
print("k =", A[0], "n =", A[1],  "e =", A[2])

Xn = 4 * A[0] + 2
m = 2 ** A[2]
max = 0

seq = list()            #Сюда пишем последовательность
norm_seq = list()       #Сюда пишем «нормированную» последовательность

for i in range(0, A[1]):
    Xn = (Xn * (Xn + 1)) % m
    seq.append(Xn)
    norm_seq.append(Xn / m)
    if max < Xn:        #для определения разрядов в покере
        max = Xn

print(" ")
print("Последовательность:")
print(seq)
print(" ")
print("Нормированная последовательность:")
print(norm_seq)
print(" ")

Ведите числа:
1) k для X0 = 2 + 4 * k
2) n для кол-ва чисел
3) e для Xn+1 = Xn * (Xn + 1) mod 2^e
 >>>>>----------------------------------------------------->
k = 3571 n = 500 e = 25
 
Последовательность:
[2777490, 21125334, 7220154, 29241054, 31111010, 2095334, 30566282, 1752558, 22808370, 25152758, 28100954, 16465150, 26691842, 12938502, 7309610, 5866510, 3405010, 61718, 17522426, 29616926, 23593634, 17756454, 6259402, 18696750, 17295986, 33064246, 32091290, 10287422, 14066754, 9651526, 28709994, 970830, 1419282, 23152982, 26178106, 22226782, 5565922, 28387686, 24051210, 2020974, 25357746, 16645494, 19689434, 16463230, 20750210, 26172806, 23986090, 25566350, 22903634, 5857686, 30958970, 32629662, 30374178, 1876390, 8313162, 29713070, 26939634, 29767094, 9638682, 31191486, 20396738, 1029574, 5589738, 27617486, 1666706, 6240726, 4470970, 27664350, 25535586, 15161830, 26023050, 20377326, 10376242, 470518, 29070938, 25505278, 28603906, 15355398, 27915818, 30586126, 14123474, 23646742, 28

# Классические критерии

## Хи-квадрат

### Подсчёт хи-квадрат

In [2]:
otr = 11    #кол-во кусков на которыек разбиваем [0,1)
len_otr = 1 / otr
D = dict()
nu = otr - 1    # степени свободы

for i in range(1, otr + 1):
    D[i] = 0

#>>>-----------------Считаем попадания в отрезки---------------------->
for element in norm_seq:
    for j in D.keys():      #определяет, в какой кусок на [0, 1) попадают элементы (element) последовательности
        if (element < j * len_otr):
            D[j] = D[j] + 1
            break

#>>>-----------------Считаем хи квадрат---------------------->
nPs = A[1] / otr
V = 0

for Ys in D.values():
    V = V + ((Ys - nPs) ** 2) / nPs

print(f"хи-квадрат = {V:.4f}")
print(f"Кол-во степеней свободы = {nu}")
print(f"nPs = {nPs:.4f}")

хи-квадрат = 7.5400
Кол-во степеней свободы = 10
nPs = 45.4545


### Оценка

Формула из книжки Кнута для $n > 30$:
$$
V_{критич-е} = n + \sqrt{2n}x_p + \frac{2}{3}x_p^2 - \frac{2}{3}+ O\left(\frac{1}{\sqrt{n}}\right)
$$

Так как было рассмотрено 10 степеней свободы, то значения можно посмотреть по таблице (но вместо этого я использовала библиотечные значения).

Нижняя граница: 3.940 – это процентная точка для $\nu$ = 10, p = 5 %

Верхняя граница: 18.307 – это процентная точка для $\nu$ = 10, p = 95 %

In [3]:
from scipy.stats import chi2

alpha = 0.05

low = chi2.ppf(alpha, nu)
up = chi2.ppf(1 - alpha, nu)

print(f"Допустимый интервал: [{low:.3f}, {up:.3f}]")

if V > low and V < up:
    print("✓ Генератор ПРОШЁЛ тест χ²")
else:
    print("✗ Генератор НЕ ПРОШЁЛ тест")

Допустимый интервал: [3.940, 18.307]
✓ Генератор ПРОШЁЛ тест χ²


## Колмогоров – Смирнов

### Подсчёт $K_n^+$ и $K_n^-$

Kn_up = $ K_n^+ \quad$ Kn_do = $ K_n^- $

Но сначала:

Kn_up = $ \underset{1 \leq i \leq n}{max} \left( \frac{i}{n} - F(X_i)\right) \quad $
Kn_do = $ \underset{1 \leq i \leq n}{max} \left( F(X_i) - \frac{i-1}{n}\right) \quad $

In [4]:
norm_seq.sort()
print(norm_seq)

Kn_up = 0
Kn_do = 0
sqr_n = A[1] ** (1/2)
n = A[1]

#>>>-----------------Ищем максимальные отклонения---------------------->
for i in range(0, n - 1):  # рассматриваем все значения кроме последнего, его дополнительно обрабатываем потом
    if norm_seq[i] == norm_seq[i+1]:
        continue

    K_temp = (i + 1) / n - norm_seq[i]
    if K_temp > Kn_up:
        Kn_up = K_temp

    K_temp = norm_seq[i] - i / n
    if K_temp > Kn_do:
        Kn_do = K_temp

# Обработка последнего значения
K_temp = 1 - norm_seq[-1]
if K_temp > Kn_up:
    Kn_up = K_temp

K_temp = norm_seq[-1] - (n - 1) / n
if K_temp > Kn_do:
    Kn_do = K_temp

#>>>-----------------Приводим к «подобающему» виду---------------------->
Kn_up = sqr_n * Kn_up
Kn_do = sqr_n * Kn_do

#>>>-----------------Выводим---------------------->
print(f"K+: {Kn_up:.10f}")
print(f"K-: {Kn_do:.10f}")

[0.0018393397331237793, 0.0024250149726867676, 0.005626857280731201, 0.006152927875518799, 0.008727610111236572, 0.009116709232330322, 0.009941518306732178, 0.014022529125213623, 0.015597760677337646, 0.01906222105026245, 0.02251964807510376, 0.027905523777008057, 0.028385818004608154, 0.02893298864364624, 0.030683696269989014, 0.03127342462539673, 0.03719216585159302, 0.037468135356903076, 0.04229789972305298, 0.04617196321487427, 0.046190083026885986, 0.049671709537506104, 0.049767911434173584, 0.05082923173904419, 0.05151158571243286, 0.05223029851913452, 0.0537148118019104, 0.05592077970504761, 0.05599242448806763, 0.05860799551010132, 0.05922800302505493, 0.05935138463973999, 0.060229718685150146, 0.06116431951522827, 0.0615573525428772, 0.06244581937789917, 0.06315678358078003, 0.06702619791030884, 0.06722551584243774, 0.0683097243309021, 0.07049626111984253, 0.07073909044265747, 0.07198518514633179, 0.07328373193740845, 0.07865697145462036, 0.0810423493385315, 0.0813184380531311

### Оценка

Формула из книжки Кнута для $n > 30$:
$$
K_{критич-е} = y_p - \frac{1}{6}n^{1/2} + O(\frac{1}{n}), \quad где \quad y_p^2 = \frac{1}{2} ln\left( \frac{1}{1-p} \right)
$$
Возьму $y_p =  1.2239$ для $p = 95 \% $

In [5]:
y_p = 1.2239
K_crit = y_p - 1/(6 * (n ** 2))
print(f"K_крит = {K_crit:.4f}")

if Kn_up < K_crit and Kn_do < K_crit:
    print("✓ Генератор ПРОШЁЛ тест Колмогорова-Смирнова")
else:
    print("✗ Генератор НЕ ПРОШЁЛ тест")

K_крит = 1.2239
✓ Генератор ПРОШЁЛ тест Колмогорова-Смирнова


# Статистические критерии

## Покер критерий

In [6]:
from collections import Counter

#>>>-----------------Преобразуем всё в одну длинную строку цифр---------------------->
str_seq = str()
digits = 5 #берем пять максимум младших разрядов каждого числа

#sort_seq = sorted(seq)
if len(str(max)) < digits: #но если самое большое число последовательности имеет менее 5 разрадова, то берем менее 5 разрядов
    digits = len(str(max))

for elem in seq:
    if len(str(elem)) < digits:
        for i in range(1, digits):
            if elem < 10 ** i:
                str_seq = str_seq + '0' * (digits - i) + str(elem)
                break
    else:
        str_seq += str(elem)[-digits:]
    
#print(str_seq)
#print(seq)

#>>>-----------------Выдаём на руки и считаем комбинации---------------------->
hands = n * digits // 5     #кол-во рук
categ = {               #категории
    'Все разные': 0,
    'Одна пара': 0,
    'Две пары или тройка': 0,      #AABBC AAABC
    'Фулл хаус или каре': 0,     #AAABB AAAAB
    'Все одинаковые': 0
}

for i in range(0, len(str_seq), 5):
    one_hand = str_seq[i:i+5]
    counts = sorted(Counter(one_hand).values(), reverse=True)
    
    if counts == [5]:
        categ['Все одинаковые'] += 1
    elif counts == [4, 1] or counts == [3, 2]:
        categ['Фулл хаус или каре'] += 1
    elif counts == [3, 1, 1] or counts == [2, 2, 1]:
        categ['Две пары или тройка'] += 1
    elif counts == [2, 1, 1, 1]:
        categ['Одна пара'] += 1
    else:  # [1, 1, 1, 1, 1]
        categ['Все разные'] += 1

#print(str_seq)
#print(seq)
print(categ)
print(f"Всего рук: {hands:.0f}")

#>>>-----------------Сравниваем с теорией (считаем χ²)---------------------->
teor = {                #теоретические вероятности «раскладок»
    'Все разные': 0.3024,
    'Одна пара': 0.5040,
    'Две пары или тройка': 0.1080 + 0.0720,
    'Фулл хаус или каре': 0.0090 + 0.0045,
    'Все одинаковые': 0.0001
}

V = 0
for key in categ.keys():
    nPs = teor[key] * hands
    Ys = categ[key]
    V = V + ((Ys - nPs) ** 2) / nPs

nu = len(categ) - 1
print(f"хи-квадрат = {V:.4f}")
print(f"Кол-во степеней свободы = {nu}")


{'Все разные': 155, 'Одна пара': 266, 'Две пары или тройка': 76, 'Фулл хаус или каре': 3, 'Все одинаковые': 0}
Всего рук: 500
хи-квадрат = 5.1844
Кол-во степеней свободы = 4


### Оценка

In [7]:
alpha = 0.05

low = chi2.ppf(alpha, nu)
up = chi2.ppf(1 - alpha, nu)

print(f"Допустимый интервал: [{low:.3f}, {up:.3f}]")

if V > low and V < up:
    print("✓ Генератор ПРОШЁЛ тест")
else:
    print("✗ Генератор НЕ ПРОШЁЛ тест")

Допустимый интервал: [0.711, 9.488]
✓ Генератор ПРОШЁЛ тест


## Критерий промежутков между днями рождений

Используем начальные данные (k, e для $2^e$) для последовательности, которые вводили выше, но теперь гененрируем не n чисел, а, скажем 1000 * n, и для каждых n чисел ищем своё R.

### m и n как у Кнута

В книжке Кнута: $m = 2^{25}$, n = 512, поэтому сначала протестируем генератор с этими входными данными.

In [8]:
m1 = 2 ** 25
n1 = 512
k1 = A[0]        #Вот эта штука определяет seed, то есть X0. Возьму от своей последовательности
Xn = 4 * k1 + 2

pieces = 1000           #проанализируем 1000 кусков по 512 чисел
list_of_R = list()      #а сюда запишем все полученные R с этих кусков

for i in range(0, pieces):
    #>>>-----------------Получаем «временную» последовательность (кусок)---------------------->
    temp_seq = list()           #Сюда пишем последовательность (вернее очередной кусок в 512 символов)
    for i in range(0, n1):      #Делаем очередной кусок для анализа
        Xn = (Xn * (Xn + 1)) % m1
        temp_seq.append(Xn)

    #>>>-----------------Получаем последовательность промежутков---------------------->
    sort_temp_seq = sorted(temp_seq)
    prom = list()               # последовательность промежутков между «днями»

    for i in range(0, len(temp_seq) - 1):
        prom.append(sort_temp_seq[i+1] - sort_temp_seq[i])
    prom.append(sort_temp_seq[0] + m1 - sort_temp_seq[-1])

    #>>>-----------------Получаем R для данного куска---------------------->
    prom.sort() #это последовательность S_j
    R = 0

    for j in range(0, len(prom) - 1):
        if prom[j] == prom[j + 1]:
            R += 1
    list_of_R. append(R)

#>>>-----------------Посчитываем R по категориям---------------------->
R_exp = {   #сюда запишем эмпирические значения R
    0: 0,   #кол-во R равных 0
    1: 0,   #кол-во R равных 1
    2: 0,
    3: 0
}

for elem in list_of_R:
    if elem == 0:
        R_exp[0] += 1
    elif elem == 1:
        R_exp[1] += 1
    elif elem == 2:
        R_exp[2] += 1
    else:
        R_exp[3] += 1

#>>>-----------------Даём оценку для R с помощью χ²---------------------->

R_teor = {              #Теоретические значения R из книжки Кнута
    0: 0.368801577,     #можно было бы списком, но словарь нагляднее
    1: 0.369035243,
    2: 0.183471182,
    3: 0.078691997      #3 и более
}  

alpha = 0.05
nu1 = len(R_teor) - 1

low = chi2.ppf(alpha, nu1)
up = chi2.ppf(1 - alpha, nu1)

print(f"Допустимый интервал: [{low:.3f}, {up:.3f}]")

V = 0
for key in R_teor.keys():
    nPs = R_teor[key] * pieces
    Ys = R_exp[key]
    V = V + ((Ys - nPs) ** 2) / nPs

if V > low and V < up:
    print("✓ Генератор ПРОШЁЛ тест")
else:
    print("✗ Генератор НЕ ПРОШЁЛ тест")

#>>>-----------------Выводим дополнительно экспериментальные значения R---------------------->
print("\n>>>>>----------------------------------------------------->")
print("Полученные R:")
print(R_exp)

Допустимый интервал: [0.352, 7.815]
✗ Генератор НЕ ПРОШЁЛ тест

>>>>>----------------------------------------------------->
Полученные R:
{0: 20, 1: 73, 2: 150, 3: 757}


### m и n как у меня выше

In [9]:
import math

print(" >>>>>-----------------------Напомню------------------------------>")
print("k =", A[0], "n =", A[1],  "e =", A[2])

Xn = 4 * A[0] + 2
m = 2 ** A[2]

pieces = 1000           #проанализируем 1000 кусков по n чисел
list_of_R = list()      #а сюда запишем все полученные R с этих кусков

for i in range(0, pieces):
    #>>>-----------------Получаем «временную» последовательность (кусок)---------------------->
    temp_seq = list()           #Сюда пишем последовательность (вернее очередной кусок в 512 символов)
    for i in range(0, A[1]):    #Делаем очередной кусок для анализа
        Xn = (Xn * (Xn + 1)) % m
        temp_seq.append(Xn)

    #>>>-----------------Получаем последовательность промежутков---------------------->
    sort_temp_seq = sorted(temp_seq)
    prom = list()               # последовательность промежутков между «днями»

    for i in range(0, len(temp_seq) - 1):
        prom.append(sort_temp_seq[i+1] - sort_temp_seq[i])
    prom.append(sort_temp_seq[0] + m - sort_temp_seq[-1])

    #>>>-----------------Получаем R для данного куска---------------------->
    prom.sort()
    #counted_prom = list(Counter(prom))  #это последовательность S_j
    R = 0

    for j in range(0, len(prom) - 1):
        if prom[j] == prom[j + 1]:
            R += 1
    list_of_R. append(R)

#>>>-----------------Посчитываем R по категориям---------------------->
R_exp = {   #сюда запишем эмпирические значения R
    0: 0,   #кол-во R равных 0
    1: 0,   #кол-во R равных 1
    2: 0,
    3: 0
}

for elem in list_of_R:
    if elem == 0:
        R_exp[0] += 1
    elif elem == 1:
        R_exp[1] += 1
    elif elem == 2:
        R_exp[2] += 1
    else:
        R_exp[3] += 1

#>>>-----------------Подсчитываем теоретические вероятности R---------------------->
R_teor = {
    0: 0,
    1: 0,
    2: 0,
    3: 0      #3 и более
}  

lamb = A[1] ** 3 / (4 * m)    # λ = (кол-во «дней рождения»)^3 / (4 * (кол-во дней в году))
R_teor[0] = lamb ** 0 * math.e ** (-lamb) / math.factorial(0)   # веро-ть по фор-ле пуассона: λ^k * e^{-1} / k!
R_teor[1] = lamb ** 1 * math.e ** (-lamb) / math.factorial(1)
R_teor[2] = lamb ** 2 * math.e ** (-lamb) / math.factorial(2)
R_teor[3] = 1 - R_teor[0] - R_teor[1] - R_teor[2]

#>>>-----------------Даём оценку для R с помощью χ²---------------------->
alpha = 0.05
nu1 = len(R_teor) - 1

low = chi2.ppf(alpha, nu1)
up = chi2.ppf(1 - alpha, nu1)

print(f"\nДопустимый интервал: [{low:.3f}, {up:.3f}]")

V = 0
for key in R_teor.keys():
    nPs = R_teor[key] * pieces
    Ys = R_exp[key]
    V = V + ((Ys - nPs) ** 2) / nPs

if V > low and V < up:
    print("✓ Генератор ПРОШЁЛ тест")
else:
    print("✗ Генератор НЕ ПРОШЁЛ тест")

#>>>-----------------Выводим дополнительно экспериментальные значения R---------------------->
print("\n>>>>>----------------------------------------------------->")
print("Полученные R:")
print(R_exp)

 >>>>>-----------------------Напомню------------------------------>
k = 3571 n = 500 e = 25

Допустимый интервал: [0.352, 7.815]
✗ Генератор НЕ ПРОШЁЛ тест

>>>>>----------------------------------------------------->
Полученные R:
{0: 23, 1: 103, 2: 166, 3: 708}


In [10]:
#>>>-----------------Проверка: считаем теоретические вер-ти R для m, n из книжки---------------------->
m0  = 2 ** 25
n0 = 512
R_teor = {
    0: 0,
    1: 0,
    2: 0,
    3: 0      #3 и более
}  

lamb = n0 ** 3 / (4 * m0)    # λ = (кол-во «дней рождения»)^3 / (4 * (кол-во дней в году))
R_teor[0] = lamb ** 0 * math.e ** (-lamb) / math.factorial(0)   # веро-ть по фор-ле пуассона: λ^k * e^{-1} / k!
R_teor[1] = lamb ** 1 * math.e ** (-lamb) / math.factorial(1)
R_teor[2] = lamb ** 2 * math.e ** (-lamb) / math.factorial(2)
R_teor[3] = 1 - R_teor[0] - R_teor[1] - R_teor[2]

print(R_teor)

{0: 0.36787944117144233, 1: 0.36787944117144233, 2: 0.18393972058572117, 3: 0.08030139707139416}
